# Planner Agent Fine-Tuning on Kaggle — Qwen2.5-7B-Instruct + QLoRA (Unsloth)

Free training on Kaggle's NVIDIA **T4** GPU. Fine-tunes the **Planner Agent** that turns
`schema` + `user_prompt` into a pipeline `config` (ADF copy + Databricks notebook stages).

## Before you press Run — Kaggle settings
1. Open in Kaggle (**Create → New Notebook**, or **File → Import Notebook** → upload this `.ipynb`).
2. Right sidebar → **Settings**:
   - **Accelerator** → **GPU T4 x2** (or **P100**).
   - **Internet** → **On** *(needs one-time phone verification, or installs/downloads fail)*.
3. **Add Input → Upload → Dataset** → upload your `.jsonl` (rows of `{schema, user_prompt, config}`). It lands under `/kaggle/input/…` and is found automatically.
4. **Run All**. First run downloads ~5 GB (red bars are normal).
5. Get the adapter from the **Output** tab (`planner_agent_lora.zip`).


## 1. Install Unsloth
If it errors, run again or **Run → Restart & clear cell outputs**, then re-run.

In [1]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 23.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.4 MB/s eta 0:00

## 2. Environment check

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
assert torch.cuda.is_available(), "No GPU. Set Settings -> Accelerator -> GPU T4 x2, then re-run."
print("PyTorch :", torch.__version__)
print("GPU     :", torch.cuda.get_device_name(0))
print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
print("bf16 supported:", torch.cuda.is_bf16_supported(), "(T4 = False -> fp16 used automatically)")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 14.6 GB
bf16 supported: True (T4 = False -> fp16 used automatically)


## 3. Imports

In [3]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import glob, json, re, gc
from datasets import Dataset
from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 4. Configuration
If you hit OOM on T4, set `BATCH_SIZE = 1` and/or lower `MAX_SEQ_LENGTH`.

In [4]:
MODEL_NAME      = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH  = 2048
LOAD_IN_4BIT    = True

LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.0

BATCH_SIZE      = 2
GRAD_ACCUM      = 4
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1
MAX_STEPS       = 0          # 0 = use epochs; set e.g. 60 for a quick test
WARMUP_STEPS    = 10
WEIGHT_DECAY    = 0.01
SEED            = 3407

OUTPUT_DIR      = "/kaggle/working/outputs"
ADAPTER_DIR     = "/kaggle/working/planner_agent_lora"

SYSTEM_PROMPT = (
    "You are the Planner Agent for a data pipeline orchestrator. Given CSV schema metadata and a "
    "user transformation request, output ONLY a JSON pipeline configuration with keys: containers, "
    "containers_to_create, datasets, stages, execution_order, num_containers, recommended_settings, "
    "editable_settings, reasoning. The first stage has type 'copy' (ADF ingest); all later stages "
    "have type 'notebook' (Databricks PySpark). Output compact JSON only, no prose."
)


## 5. Load model and tokenizer (4-bit)

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    dtype          = None,
)
print("Model loaded.")


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded.


## 6. Attach LoRA adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    max_seq_length             = MAX_SEQ_LENGTH,
    use_rslora                 = False,
    loftq_config               = None,
)
print("LoRA attached.")


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA attached.


## 7. Qwen2.5 chat template

In [7]:
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
print("Chat template set: qwen-2.5")


Chat template set: qwen-2.5


## 8. Load training data
Reads your `.jsonl` line by line (avoids nested-JSON schema inference issues). Each row must have
`schema`, `user_prompt`, and `config`. Falls back to a tiny built-in sample if nothing is attached.

In [8]:
SAMPLE_RECORDS = [
    {"schema": {"columns": ["emp_name", "department", "salary"],
                "inferred_types": {"emp_name": "string", "department": "string", "salary": "integer"},
                "row_count": 980, "size_hint": "small (< 5MB)",
                "samples": [{"emp_name": "Tom Becker", "department": "HR", "salary": "5809"}]},
     "user_prompt": "ingest data from raw into bronze; then in silver, keep rows where salary > 5000.",
     "config": {"containers": {"stage0": "raw", "stage1": "bronze", "stage2": "silver"},
                "containers_to_create": ["raw", "bronze", "silver"],
                "datasets": [{"name": "DS_Raw", "container": "raw", "role": "source"},
                             {"name": "DS_Bronze", "container": "bronze", "role": "intermediate"},
                             {"name": "DS_Silver", "container": "silver", "role": "sink"}],
                "stages": [{"name": "Ingest_Raw_To_Bronze", "type": "copy",
                            "source_dataset": "DS_Raw", "sink_dataset": "DS_Bronze", "diu": 2},
                           {"name": "Transform_Bronze_To_Silver", "type": "notebook",
                            "source_container": "bronze", "sink_container": "silver",
                            "transformations": ["processed_time = currentTimestamp()"],
                            "filter_condition": "salary > 5000", "num_workers": 0,
                            "shuffle_partitions": 4}],
                "execution_order": ["Ingest_Raw_To_Bronze", "Transform_Bronze_To_Silver"],
                "num_containers": 3,
                "recommended_settings": {"diu": 2, "num_workers": 0, "shuffle_partitions": 4,
                                         "node_type": "Standard_D4s_v3"},
                "editable_settings": {"diu": [1, 2, 4, 8, 16, 32], "num_workers": [0, 2, 4, 8, 16],
                                      "shuffle_partitions": [4, 8, 16, 32, 64],
                                      "node_type": ["Standard_D4s_v3", "Standard_DS4_v2", "Standard_D8s_v3"]},
                "reasoning": "3-stage medallion pipeline: ADF copy ingests raw->bronze; the notebook applies the filter."}},
]


def load_records():
    files = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
    if files:
        path = files[0]
        recs = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    recs.append(json.loads(line))
        print(f"Loaded {len(recs)} records from {path}")
        return recs
    print(f"No .jsonl under /kaggle/input — using {len(SAMPLE_RECORDS)} built-in sample(s).")
    return SAMPLE_RECORDS


raw_records = load_records()
print("Top-level keys of first record:", list(raw_records[0].keys()))


Loaded 5000 records from /kaggle/input/datasets/manvithnayak/planner-config-dataset-2-jsonl/planner_config_dataset.jsonl
Top-level keys of first record: ['schema', 'user_prompt', 'config']


## 9. Format into Qwen chat text
- **user** = JSON of `{schema, user_prompt}`
- **assistant** = the `config` as compact single-line JSON (clean, parseable target)

In [9]:
def build_messages(rec):
    user = json.dumps({"schema": rec["schema"], "user_prompt": rec["user_prompt"]},
                      ensure_ascii=False)
    assistant = json.dumps(rec["config"], ensure_ascii=False, separators=(",", ":"))
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]


def record_to_text(rec):
    msgs = build_messages(rec)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)


texts = [{"text": record_to_text(r)} for r in raw_records]
dataset = Dataset.from_list(texts)
print("Examples:", len(dataset))
print(dataset[0]["text"][:900])


Examples: 5000
<|im_start|>system
You are the Planner Agent for a data pipeline orchestrator. Given CSV schema metadata and a user transformation request, output ONLY a JSON pipeline configuration with keys: containers, containers_to_create, datasets, stages, execution_order, num_containers, recommended_settings, editable_settings, reasoning. The first stage has type 'copy' (ADF ingest); all later stages have type 'notebook' (Databricks PySpark). Output compact JSON only, no prose.<|im_end|>
<|im_start|>user
{"schema": {"columns": ["order_id", "region", "channel", "product", "quantity", "unit_price", "discount", "customer_email"], "inferred_types": {"order_id": "integer", "region": "string", "channel": "string", "product": "string", "quantity": "integer", "unit_price": "double", "discount": "double", "customer_email": "string"}, "row_count": 431409, "size_hint": "medium (5–50MB)", "samples": [{"order_i


## 10. Configure the trainer

In [10]:
sft_config_kwargs = dict(
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps                = WARMUP_STEPS,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = "linear",
    optim                       = "adamw_8bit",
    logging_steps               = 1,
    seed                        = SEED,
    output_dir                  = OUTPUT_DIR,
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    dataset_num_proc            = 2,
    packing                     = False,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    report_to                   = "none",
)

if MAX_STEPS and MAX_STEPS > 0:
    sft_config_kwargs["max_steps"] = MAX_STEPS
else:
    sft_config_kwargs["num_train_epochs"] = NUM_EPOCHS

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = SFTConfig(**sft_config_kwargs),
)
print("Trainer ready.")


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready.


## 11. Train only on the assistant's answer

In [11]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
print("Response-only masking applied.")


Map (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

Response-only masking applied.


## 12. Train

In [12]:
print("Peak VRAM reserved before:", round(torch.cuda.max_memory_reserved()/1024**3, 2), "GB")
trainer_stats = trainer.train()
print("Peak VRAM reserved after :", round(torch.cuda.max_memory_reserved()/1024**3, 2), "GB")
print("Training runtime         :", round(trainer_stats.metrics["train_runtime"], 1), "s")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Peak VRAM reserved before: 5.39 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.020012
2,1.033011
3,0.872716
4,0.985856
5,0.814936
6,0.638874
7,0.533234
8,0.479058
9,0.367045
10,0.334006


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-625/tokenizer_config.json.


Peak VRAM reserved after : 14.37 GB
Training runtime         : 16097.2 s


## 13. Compare base vs fine-tuned (before / after)
Runs the **same prompts** through the base model and your fine-tuned adapter, prints them side by
side, and scores both on your real contract:
- **valid-JSON rate** — output parses as JSON
- **contract rate** — has the 9 config keys; `stages[0].type=="copy"`, rest `"notebook"`;
  `num_containers == len(containers_to_create)`; `len(stages) == num_containers - 1`;
  `execution_order` equals the stage names in order

In [13]:
# held-out eval inputs (edit / add your own; ideally NOT in the training set)
EVAL_INPUTS = [
    {"schema": {"columns": ["order_id", "region", "price", "quantity"],
                "inferred_types": {"order_id": "integer", "region": "string",
                                   "price": "double", "quantity": "integer"},
                "row_count": 5000, "size_hint": "small (< 5MB)",
                "samples": [{"order_id": "1", "region": "EU", "price": "99.5", "quantity": "3"}]},
     "user_prompt": "ingest data from raw into bronze; then in silver, keep only rows where price > 50."},
    {"schema": {"columns": ["device_id", "location", "temperature"],
                "inferred_types": {"device_id": "string", "location": "string", "temperature": "double"},
                "row_count": 900000, "size_hint": "large (50-200MB)",
                "samples": [{"device_id": "D1", "location": "lab", "temperature": "42.1"}]},
     "user_prompt": "ingest from raw into bronze; then stage from bronze into silver; then in gold, "
                    "normalize location to uppercase as location_norm, keep rows where location_norm = 'RETAIL'."},
]

REQUIRED_CFG_KEYS = {"containers", "containers_to_create", "datasets", "stages", "execution_order",
                     "num_containers", "recommended_settings", "editable_settings", "reasoning"}


def generate(mdl, tok, user_content):
    FastLanguageModel.for_inference(mdl)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}]
    ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                  return_tensors="pt").to("cuda")
    out = mdl.generate(input_ids=ids, max_new_tokens=512, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


def extract_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def score(text):
    cfg = extract_json(text)
    valid = cfg is not None
    contract = False
    if valid:
        try:
            stages = cfg["stages"]
            contract = bool(
                REQUIRED_CFG_KEYS.issubset(cfg.keys())
                and stages[0]["type"] == "copy"
                and all(s["type"] == "notebook" for s in stages[1:])
                and cfg["num_containers"] == len(cfg["containers_to_create"])
                and len(stages) == cfg["num_containers"] - 1
                and cfg["execution_order"] == [s["name"] for s in stages]
            )
        except Exception:
            contract = False
    return valid, contract


prompts = [json.dumps(e, ensure_ascii=False) for e in EVAL_INPUTS]

# fine-tuned outputs (current model already has the adapter)
ft_out = [generate(model, tokenizer, p) for p in prompts]

# load a clean BASE model (no adapter)
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT, dtype=None,
)
base_tok = get_chat_template(base_tok, chat_template="qwen-2.5")
base_out = [generate(base_model, base_tok, p) for p in prompts]

for i in range(len(prompts)):
    print("=" * 84)
    print("PROMPT", i + 1, ":", EVAL_INPUTS[i]["user_prompt"][:80])
    print("-" * 84)
    print("BASE      :", base_out[i][:280])
    print("-" * 84)
    print("FINE-TUNED:", ft_out[i][:280])
    print()


def rate(outs, idx):
    vals = [score(o)[idx] for o in outs]
    return 100.0 * sum(vals) / len(vals)


bv, fv = rate(base_out, 0), rate(ft_out, 0)
bc, fc = rate(base_out, 1), rate(ft_out, 1)
print("=" * 56)
print(f"{'metric':<22}{'base':>10}{'fine-tuned':>14}{'delta':>10}")
print("-" * 56)
print(f"{'valid-JSON rate %':<22}{bv:>10.0f}{fv:>14.0f}{fv-bv:>+10.0f}")
print(f"{'contract rate %':<22}{bc:>10.0f}{fc:>14.0f}{fc-bc:>+10.0f}")

del base_model
gc.collect(); torch.cuda.empty_cache()
FastLanguageModel.for_inference(model)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT 1 : ingest data from raw into bronze; then in silver, keep only rows where price > 5
------------------------------------------------------------------------------------
BASE      : {
  "containers": {
    "bronze": {},
    "silver": {}
  },
  "containers_to_create": [
    "bronze",
    "silver"
  ],
  "datasets": {
    "raw": {
      "type": "csv",
      "path": "/raw",
      "schema": {
        "columns": ["order_id", "region", "price", "quantity"],
      
------------------------------------------------------------------------------------
FINE-TUNED: {"containers":{"stage0":"raw","stage1":"bronze","stage2":"silver"},"containers_to_create":["raw","bronze","silver"],"datasets":[{"name":"DS_Raw","container":"raw","role":"source"},{"name":"DS_Bronze","container":"bronze","role":"intermediate"},{"name":"DS_Silver","container":"sil

PROMPT 2 : ingest from raw into bronze; then stage from bronze into silver; then in gold, n
-----------------------------------------------------------

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## 14. Save the adapter to Kaggle output

In [14]:
import shutil
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)
print("Download:", ADAPTER_DIR + ".zip  (Output tab)")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/planner_agent_lora/tokenizer_config.json.


Saved adapter to /kaggle/working/planner_agent_lora
Download: /kaggle/working/planner_agent_lora.zip  (Output tab)


## 15. Reloading the adapter later
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/kaggle/working/planner_agent_lora",
    max_seq_length = 2048,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
```
